# 03 - EDA: RSNA Pneumonia Detection Challenge

Misma estructura que los notebooks anteriores. Lo que cambia aquí: las imágenes son **DICOM** (`.dcm`), no JPG/PNG, así que la carga usa `pydicom` en vez de PIL (`eda_utils.probe_dicom` / `dicom_metadata`), y por primera vez hay metadata real de paciente (edad, sexo, vista) en la cabecera de cada archivo.

## 0. Configuración

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

tqdm.pandas()

DATASET_NAME = "RSNA_Pneumonia"
DATASET_DIR = Path("../Datasets/RSNA_Pneumonia")
TRAIN_DIR = DATASET_DIR / "stage_2_train_images"
TEST_DIR = DATASET_DIR / "stage_2_test_images"
CLASSINFO_CSV = DATASET_DIR / "stage_2_detailed_class_info.csv"
LABELS_CSV = DATASET_DIR / "stage_2_train_labels.csv"
SUBMISSION_CSV = DATASET_DIR / "stage_2_sample_submission.csv"

# class de origen -> clase objetivo del proyecto.
# "No Lung Opacity / Not Normal" se excluye: son radiografías ANORMALES pero sin opacidad de
# neumonía (otros hallazgos) — mismo tipo de categoría ambigua que "sick_but_no_tb" en TBX11K.
CLASS_MAP = {
    "Normal": "Normal",
    "Lung Opacity": "Pneumonia",
}
EXCLUDED_CLASSES = {"No Lung Opacity / Not Normal"}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("../resultados/eda") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HASH_DIR = Path("../Datasets/_eda_hashes")
HASH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Inventario básico
Al igual que en TBX11K, aquí también hay que revisar si el CSV cuenta por imagen o por caja delimitadora antes de confiar en ningún conteo.

In [ ]:
print(f"Archivos físicos en stage_2_train_images/: {len(list(TRAIN_DIR.glob('*.dcm')))}")
print(f"Archivos físicos en stage_2_test_images/:  {len(list(TEST_DIR.glob('*.dcm')))}")

classinfo = pd.read_csv(CLASSINFO_CSV)
labels = pd.read_csv(LABELS_CSV)
submission = pd.read_csv(SUBMISSION_CSV)
print(f"Filas en stage_2_detailed_class_info.csv: {len(classinfo)}")
print(f"Nombres de paciente únicos: {classinfo['patientId'].nunique()}")
print(f"Filas 'de más' por múltiples cajas delimitadoras: {classinfo['patientId'].duplicated().sum()}")
print("¿En qué clase ocurre esto?")
classinfo.groupby("class")["patientId"].apply(lambda s: s.duplicated().sum())

### `stage_2_test_images/` (3,000 archivos) — igual de inutilizable que `test/` en TBX11K

`stage_2_sample_submission.csv` no son etiquetas reales: es la plantilla de envío de la competencia de Kaggle, con una predicción de relleno idéntica para todos (`"0.5 0 0 100 100"`). Se verifica que coincide 1 a 1 con los 3,000 archivos de `stage_2_test_images/` — es el test set privado del reto original, sin ninguna etiqueta real disponible. **Se excluye por completo**, igual que se hizo con `test/` en TBX11K.

In [ ]:
test_files = {p.stem for p in TEST_DIR.glob("*.dcm")}
submission_ids = set(submission["patientId"])
print(f"Coinciden exactamente: {test_files == submission_ids}")
print(f"Valores únicos de PredictionString (para confirmar que es una plantilla, no etiquetas reales): "
      f"{submission['PredictionString'].nunique()}")

## 2. Esquema de etiquetas y mapeo a clases objetivo
Se deduplica por `patientId` primero (una imagen = una fila), se excluye la clase ambigua, y se mapea el resto.

In [ ]:
classinfo_unique = classinfo.drop_duplicates(subset="patientId").copy()

excluded = classinfo_unique[classinfo_unique["class"].isin(EXCLUDED_CLASSES)]
print(f"Excluidas por ser 'No Lung Opacity / Not Normal' (anormales pero no neumonía): {len(excluded)}")

classinfo_unique["target_class"] = classinfo_unique["class"].map(CLASS_MAP)
unmapped = classinfo_unique[classinfo_unique["target_class"].isna() & ~classinfo_unique["class"].isin(EXCLUDED_CLASSES)]
if len(unmapped):
    print("Clases sin mapear (revisar CLASS_MAP):", sorted(unmapped["class"].unique()))

df = classinfo_unique.dropna(subset=["target_class"]).copy()
df["path"] = df["patientId"].apply(lambda pid: str(TRAIN_DIR / f"{pid}.dcm"))
df["split"] = "train"  # RSNA no trae train/val propio; el split para el proyecto se decide en la fusión
print(f"Total utilizable (Normal + Pneumonia): {len(df)}")

## 3. Conteo real por clase objetivo

In [ ]:
totales = df["target_class"].value_counts().sort_index()
print(totales)
totales.to_frame("n").to_csv(OUTPUT_DIR / "conteo_por_clase.csv")
ax = totales.plot(kind="bar", figsize=(6, 4), color="#8172B2")
ax.set_title(f"{DATASET_NAME}: total de imágenes por clase")
ax.set_ylabel("Número de imágenes")
for i, v in enumerate(totales):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conteo_por_clase.png", dpi=120)
plt.show()

## 4. Calidad e integridad

Con DICOM conviene decodificar cada archivo **una sola vez** (es la parte lenta) y reutilizar ese array para todo: dimensiones, heurísticas de calidad y hash perceptual. Esto tarda varios minutos para las ~20 mil imágenes (se hizo la prueba de tiempo antes de escribir esta celda: ~13 ms/imagen ≈ 4-5 minutos en total).

In [ ]:
widths, heights, errors = [], [], []
content_ratios, glare_fractions, phashes = [], [], []

for p in tqdm(df["path"], desc="Decodificando DICOM (una vez por archivo)"):
    arr, err = eu.probe_dicom(p)
    if err is not None:
        widths.append(None); heights.append(None); errors.append(err)
        content_ratios.append(None); glare_fractions.append(None); phashes.append(None)
        continue
    h, w = arr.shape[:2]
    widths.append(w); heights.append(h); errors.append(None)
    thumb = eu.array_to_thumbnail_gray(arr)
    content_ratios.append(eu.content_ratio_from_array(thumb))
    glare_fractions.append(eu.glare_fraction_from_array(thumb))
    phashes.append(eu.phash_from_array(thumb))

df["width"] = widths
df["height"] = heights
df["error"] = errors
df["content_ratio"] = content_ratios
df["glare_fraction"] = glare_fractions
df["phash"] = phashes

corrupt = df[df["error"].notna()]
print(f"Imágenes corruptas / no decodificables: {len(corrupt)}")
corrupt[["path", "error"]]

In [ ]:
print("Dimensiones (ancho x alto):")
df[["width", "height"]].describe()

### 4.1 Candidatas a excluir por calidad de imagen (mismo criterio que las fuentes anteriores)
Ya calculado en la celda anterior en la misma pasada (no hace falta releer los archivos).

In [ ]:
CONTENT_RATIO_THRESHOLD = 0.55
GLARE_THRESHOLD = 0.15

calidad = df[df["error"].isna()].copy()
calidad["flag_bordes"] = calidad["content_ratio"] < CONTENT_RATIO_THRESHOLD
calidad["flag_brillo"] = calidad["glare_fraction"] > GLARE_THRESHOLD
calidad["motivo"] = calidad.apply(
    lambda r: ", ".join(
        m for m, f in [("bordes/recorte incompleto", r["flag_bordes"]), ("posible brillo/reflejo", r["flag_brillo"])] if f
    ),
    axis=1,
)
sospechosas = calidad[calidad["flag_bordes"] | calidad["flag_brillo"]].copy()
print(f"Candidatas marcadas para revisión manual: {len(sospechosas)} de {len(calidad)} "
      f"({100 * len(sospechosas) / len(calidad):.1f}%)")
sospechosas["target_class"].value_counts()

In [ ]:
sospechosas[["path", "target_class", "content_ratio", "glare_fraction", "motivo"]].to_csv(
    OUTPUT_DIR / "candidatas_baja_calidad.csv", index=False
)

def _dicom_loader(path):
    arr, _ = eu.probe_dicom(path)
    return arr

eu.show_flagged(sospechosas, save_path=OUTPUT_DIR / "candidatas_baja_calidad_muestra.png", loader=_dicom_loader)

## 5. Muestra visual por clase

In [ ]:
eu.show_samples(df[df["error"].isna()], save_path=OUTPUT_DIR / "muestra_por_clase.png", loader=_dicom_loader)

## 6. Riesgo de solapamiento / duplicados
El `md5` de un DICOM solo detecta copias byte-idénticas del propio `.dcm` (poco probable entre fuentes). El `phash` (ya calculado en la Sección 4) es el que de verdad importa aquí: RSNA reutiliza imágenes que también podrían aparecer, ya convertidas a JPG/PNG, en otras fuentes de neumonía del proyecto (Chest_x_ray, COVID-QU-Ex) — eso solo lo detecta un hash perceptual, no un hash exacto de archivo.

In [ ]:
df["md5"] = df["path"].progress_apply(eu.md5_file)
n_dupes = df["md5"].duplicated().sum()
print(f"Duplicados exactos (mismo archivo .dcm) dentro de este dataset: {n_dupes}")
print(f"Valores de phash duplicados dentro de este mismo dataset: {df['phash'].duplicated().sum()}")

In [ ]:
df[["path", "md5", "phash", "target_class", "split"]].assign(source_dataset=DATASET_NAME).to_csv(
    HASH_DIR / f"{DATASET_NAME}_md5.csv", index=False
)

## 7. Metadata demográfica / poblacional

**Primera fuente con metadata real de paciente por imagen**, leída directamente de la cabecera DICOM (no depende de ningún CSV externo). Es una lectura rápida (sin decodificar píxeles), se hace sobre las ~20 mil imágenes completas, no una muestra.

In [ ]:
meta_records = [eu.dicom_metadata(p) for p in tqdm(df["path"], desc="Leyendo metadata DICOM")]
meta_df = pd.DataFrame(meta_records)
df = pd.concat([df.reset_index(drop=True), meta_df.reset_index(drop=True)], axis=1)

df["PatientAge"] = pd.to_numeric(df["PatientAge"], errors="coerce")
print("Edad del paciente:")
display(df["PatientAge"].describe())
print()
print("Sexo del paciente:")
display(df["PatientSex"].value_counts())
print()
print("Vista de proyección:")
display(df["ViewPosition"].value_counts())
print()
print("Modalidad / parte del cuerpo (deberían ser constantes):")
print(df["Modality"].value_counts().to_dict(), df["BodyPartExamined"].value_counts().to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.boxplot(column="PatientAge", by="target_class", ax=axes[0])
axes[0].set_title("Edad por clase")
pd.crosstab(df["target_class"], df["PatientSex"]).plot(kind="bar", ax=axes[1])
axes[1].set_title("Sexo por clase")
plt.suptitle("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "demografia_por_clase.png", dpi=120)
plt.show()

Esto permite, por primera vez en este proyecto, documentar con datos reales (no solo por lo que diga el paper de origen) la población de una de las fuentes de Pneumonia — relevante para la discusión sobre heterogeneidad de población entre clases que ya se identificó como limitación en el artículo V2.

## 8. Veredicto

- Mismo patrón que TBX11K: el CSV cuenta por caja delimitadora, no por imagen — ya deduplicado por `patientId` antes de contar.
- `stage_2_test_images/` (3,000 imágenes) confirmado como no utilizable (plantilla de Kaggle, sin etiquetas reales) — igual que `test/` en TBX11K.
- Conteo real utilizable: Pneumonia (`Lung Opacity`) 6,012 / Normal 8,851. 11,821 imágenes `No Lung Opacity / Not Normal` correctamente excluidas (anormales pero no neumonía).
- Primera fuente con metadata demográfica real por imagen (edad, sexo, vista) extraída directamente del DICOM — no hay que asumirla de un paper externo como en Chest_x_ray.
- Se calculó **hash perceptual (phash)**, no solo MD5 exacto, porque aquí es donde el riesgo de solapamiento entre formatos distintos (DICOM vs JPG/PNG en otras fuentes) se vuelve real — queda listo para el cruce en el notebook de consolidación contra Chest_x_ray y, sobre todo, contra el `Non-COVID`/Pneumonia de COVID-QU-Ex.
- Aporta como techo real: hasta 6,012 Pneumonia adicionales (población adulta, con metadata real) y hasta 8,851 Normal — pendiente de saber cuánto de esto es realmente nuevo tras el cruce de hashes.